# GraphGenerator on ZINC

This notebook demonstrates the two-stage `GraphGenerator`: first generate a new interpretation graph, then instantiate base molecules conditionally from nearby ZINC examples.

In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from collections import Counter

from nsppk import NSPPK
from sklearn.ensemble import RandomForestClassifier

from abstractgraph.display import display, display_decomposition_graph, display_mappings
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator
from abstractgraph_ml.feasibility import FeasibilityEstimator, FeasibilityEstimatorFeatureCannotExist



In [3]:
def draw(graph, decomposition_function, *, nbits=11, label_mode="operator_hash", size=(12, 6), n_elements_per_row=8):
    ag = graph_to_abstract_graph(
        graph,
        decomposition_function=decomposition_function,
        nbits=nbits,
        label_mode=label_mode,
    )
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag

---

In [4]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 3000
min_num_nodes = 30
max_num_nodes = 40

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


dataset: zinc_250k
n_graphs: 3000
node_range: [30, 40]


In [5]:
label_mode = "histogram_values" #label_mode: str = "operator_hash" (default) or "histogram" or "histogram_values" for AbstractGraph node labeling.

nbits = 14

cycle_tree = add(
    compose(name("cycle"), cycle()),
    compose(name("tree"), tree()),
)
decomposition_function = compose(intersection_edges(), cycle_tree)

edge_vectorizer = NSPPK(radius=1, distance=3, connector=1, nbits=12, dense=True, parallel=True)
edge_graph_estimator = GraphEstimator(
    transformer=edge_vectorizer,
    estimator=RandomForestClassifier(
        n_estimators=80,
        random_state=0,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
feasibility_kwargs = dict(
    nbits=19,
    parallel=True,
    backend="loky",
    n_jobs=-1,
)
partial_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
partial_feasibility_estimator = FeasibilityEstimator(partial_feasibility_estimators)

final_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
final_feasibility_estimator = FeasibilityEstimator(final_feasibility_estimators)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------

edge_generator = EdgeGenerator(
    partial_feasibility_estimator=partial_feasibility_estimator,
    final_feasibility_estimator=final_feasibility_estimator,
    graph_estimator=edge_graph_estimator,
    n_negative_per_positive=3,
    n_replicates=2,
    beam_size=3,
    max_restarts=2,
    fit_n_jobs=-1,
    fit_backend="loky",
    seed=0,
)

graph_vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, dense=True, parallel=True)
conditional_generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=graph_vectorizer,
    n_jobs=1,
)

generator = GraphGenerator(
    edge_generator=edge_generator,
    conditional_generator=conditional_generator,
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    interpretation_neighbor_vectorizer=NSPPK(radius=1, distance=3, connector=1, nbits=12, dense=True, parallel=True),
    seed=None,
    debug=True,
    require_new_interpretation_graph=True,
    max_same_interpretation_retries=3,
)


In [6]:
generator.store(graphs)

In [ ]:
n_samples = 4
n_instances_per_sample = 3
generated_graphs = generator.sample(
    n_samples=n_samples,
    n_interpretation_neighbors=10,
    n_conditional_neighbors=100,
    n_instances_per_sample=n_instances_per_sample,
    interpretation_edge_removal_size=0,  # 0 bypasses edge generation; 1 removes all interpretation edges before regrowth.
    random_state=None,
    conditional_generate_kwargs=dict(
        random_state=None,
        max_backtracks=2000,
        max_attempts_per_sample=6,
        require_signature_coverage=True,
    ),
)

generated_groups = [
    generated_graphs[i : i + n_instances_per_sample]
    for i in range(0, len(generated_graphs), n_instances_per_sample)
]

print(f"generated molecules: {len(generated_graphs)}")
print("attempted seed indices:", generator.last_sampled_indices_)
print("successful seed indices:", generator.last_successful_sampled_indices_)
print("successful interpretation targets:", len(generator.last_generated_interpretation_graphs_))
if not generated_graphs:
    print("No molecules generated; inspect warnings and try a larger neighborhood or dataset slice.")


[DEBUG] event=fit_dictionaries
[DEBUG]   bucket_keys=338
[DEBUG]   components=931
[DEBUG]   interpretation_pool=100
[DEBUG]   inv_freq_keys=512
[DEBUG]   inv_keys=512
[DEBUG]   skipped_missing_anchor_components=0
[DEBUG] event=fit_distributions_index
[DEBUG]   bucket_size=min/mean/max=(1/2.75/124)
[DEBUG]   inv_members=min/mean/max=(1/2.28/124)
[DEBUG] event=fit_distributions_components
[DEBUG]   anchors_per_port=min/mean/max=(1/1.11/2)
[DEBUG]   component_degree=min/mean/max=(1/1.79/5)
[DEBUG]   ports_per_component=min/mean/max=(1/1.79/5)
[DEBUG] event=fit_anchors
[DEBUG]   inv_multiplicity_hist={1: 421, 2: 91}
[DEBUG]   unique_anchor_types=3
[DEBUG] event=fit_anchor_hist
[DEBUG]   anchors_per_base_subgraph_hist={1: 375, 2: 341, 3: 110, 4: 73, 5: 29, 6: 3}
[DEBUG] event=fit_top_buckets
[DEBUG]   top_buckets=
[DEBUG]     [{'key': (18762, 1), 'size': 124}, {'key': (389136, 1), 'size': 58},
[DEBUG]      {'key': (291049, 1), 'size': 56}, {'key': (110962, 2), 'size': 44},
[DEBUG]      {'ke

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (seed_idx,seed_graph,seed_interpretation_graph,generated_interpretation_graph,generated_instances,) in enumerate(zip(generator.last_successful_sampled_indices_,generator.last_seed_graphs_,generator.last_seed_interpretation_graphs_,generator.last_generated_interpretation_graphs_,generated_groups,)):
        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)
        
        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (seed_idx,seed_graph,seed_interpretation_graph,generated_interpretation_graph,generated_instances,) in enumerate(zip(generator.last_successful_sampled_indices_,generator.last_seed_graphs_,generator.last_seed_interpretation_graphs_,generator.last_generated_interpretation_graphs_,generated_groups,)):
        print("=" * 120)
        print(f"sample {sample_idx} | seed index {seed_idx}")

        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)

        print("seed interpretation graph")
        display([seed_interpretation_graph], size=(5, 4))

        print("generated interpretation graph")
        display([generated_interpretation_graph], size=(5, 4))

        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)
        for generated_instance in generated_instances:
            draw(generated_instance, decomposition_function=decomposition_function, nbits=nbits, label_mode=label_mode)
